# Module 04.10 — Kibana Spaces (`space`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.10 Kibana Spaces (`space`)

Spaces are Kibana's **multi-tenancy mechanism**: each space is an isolated namespace with
its own set of saved objects. A dashboard in Space A is invisible from Space B; a Data View
created in Space A does not exist in Space B. Users can be restricted to one or more spaces
via role-based access control.

Under the hood, every saved object that belongs to a non-default space has its Elasticsearch
document stored in a dedicated `.kibana_<space-id>` index (or a shared `.kibana` index with
a `namespaces` field, depending on whether the type uses the legacy or modern namespace
model). The `space` saved object itself — the metadata record for the space — is always
stored in the `default` space's `.kibana` index.

When you take a `kibana` feature-state snapshot, **every space is included**: all the
`.kibana*` indices are snapshotted as a unit. This is important for recovery scenarios —
you do not need to snapshot spaces individually. The flip side is that you cannot
selectively restore a single space from a feature-state snapshot without also restoring
all other spaces. If you need per-space portability, use the **Saved Objects UI import/export**
or the **Copy to Space** feature instead.

When you snapshot and restore the `kibana` feature state, spaces are restored first
(they have no dependencies), and then each space's objects are restored into it.

In [ ]:
heading('4.10 Spaces — create a second space')

import requests as req

# Create a new space
space_resp = req.post(
    f'{KIBANA_HOST}/api/spaces/space',
    auth=('elastic', ELASTIC_PASSWORD),
    headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
    json={
        'id': 'training-space',
        'name': 'Training Space',
        'description': 'Isolated space for snapshot training exercises',
        'color': '#AA4433',
        'initials': 'TS',
    },
)
if space_resp.status_code in (200, 409):
    success('Space created (or already exists): training-space')
else:
    warn(f'Space creation: {space_resp.status_code} {space_resp.text}')

In [ ]:
# List all spaces
spaces = kibana_get('/api/spaces/space')
t = Table('ID', 'Name', 'Description')
for s in spaces:
    t.add_row(s['id'], s['name'], s.get('description', '—')[:50])
console.print(t)

info('Spaces are stored as saved objects of type "space" in the default space.')
info('Objects created in training-space use path: /s/training-space/api/...')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/management/kibana/spaces', 'Stack Management → Spaces')
# Switch to the new training space using the space selector (top-left avatar):
kibana_link('/s/training-space/app/home', 'Training Space — home page')

In [ ]:
# Create a data view inside the new space
heading('Create a data view inside training-space')

space_dv = kibana_post('/s/training-space/api/data_views/data_view', {
    'data_view': {
        'title': 'training-space-data-*',
        'name': 'Training Space Data View',
    },
})
space_dv_id = space_dv['data_view']['id']
success(f'Data view created in training-space: {space_dv_id}')

# Show it is NOT visible in the default space
default_dvs = find_saved_objects('index-pattern', space='default')
space_dvs = find_saved_objects('index-pattern', space='training-space')
console.print(f'  Data views in default space : {len(default_dvs)}')
console.print(f'  Data views in training-space: {len(space_dvs)}')

In [ ]:
heading('Space — snapshot → delete → restore')

import requests as req

# Ensure the training space exists (re-runnable)
spaces = kibana_get('/api/spaces/space')
if not any(s['id'] == 'training-space' for s in spaces):
    warn('training-space missing — recreating')
    req.post(f'{KIBANA_HOST}/api/spaces/space',
        auth=('elastic', KIBANA_PASSWORD),
        headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
        json={'id': 'training-space', 'name': 'Training Space', 'description': 'Isolated space for snapshot exercises', 'color': '#e0629d'},
    ).raise_for_status()
    success('training-space recreated')
else:
    info('training-space already exists')

snap_delete_restore_cycle('snap-space-test', 'Kibana Space')

# Delete the space
del_resp = req.delete(f'{KIBANA_HOST}/api/spaces/space/training-space',
    auth=('elastic', KIBANA_PASSWORD),
    headers={'kbn-xsrf': 'true'},
)
del_resp.raise_for_status()
warn('Accidentally deleted training-space')
spaces_after = kibana_get('/api/spaces/space')
assert not any(s['id'] == 'training-space' for s in spaces_after), 'Space should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-space-test')
time.sleep(3)

spaces_restored = kibana_get('/api/spaces/space')
assert any(s['id'] == 'training-space' for s in spaces_restored), 'Space should be restored'
success('training-space successfully restored!')
kibana_link('/app/management/kibana/spaces', 'Stack Management → Kibana → Spaces — verify space is back')